In [1]:
from __future__ import annotations
import json
from pathlib import Path
from vllm_utils import *
from drgrpo_grader import r1_zero_reward_fn

## Start vllm engine

In [2]:
model = "allenai/OLMo-2-0425-1B"
# model = "Qwen/Qwen2.5-0.5B"
# model = "EleutherAI/pythia-410m-deduped"


llm = VLLMServer(model, gpu=1, gpu_memory_utilization=0.8)
llm.start()

(EngineCore pid=423108) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=423108) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  2.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  3.68it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  3.37it/s]
(EngineCore pid=423108) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:00<00:00, 105.22it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 131.93it/s]
(AP

(APIServer pid=422663) INFO:     127.0.0.1:56798 - "GET /health HTTP/1.1" 200 OK


## Loading data

In [3]:
def load_jsonl(path):
    with Path(path).open() as f:
        return [json.loads(line) for line in f]

data_path = Path("../data/gsm8k")
train_data = load_jsonl(data_path / "train.jsonl")
test_data = load_jsonl(data_path / "test.jsonl")


## Evaluation functions

In [4]:
def question_only_prompt(question):
    return f"Question: {question}\nAnswer: Let's think step by step."

def r1zero_prompt(question):
    prompt = f"A conversation between User and Assistant. The User asks a question, and the Assistant solvesit. The Assistant first thinks about the reasoning process in the mind and then provides theUser with the answer. The reasoning process is enclosed within <think> </think> and theanswer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoningprocess here </think> <answer> answer here </answer>.User: {question} Assistant: <think>"
    return prompt


def r1zero_3shot_prompt(question):
    prompt = f"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>\nUser: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAssistant: <think> There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. So the answer is 6. </think> <answer> 6 </answer>\nUser: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAssistant: <think> There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. So the answer is 5. </think> <answer> 5 </answer>\nUser: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAssistant: <think> Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. So the answer is 39. </think> <answer> 39 </answer>\nUser: {question}\nAssistant: <think>"
    return prompt


def evaluate_llm(llm, reward_fn, prompt_fn, problems, eval_sampling_params) -> None:
    n = eval_sampling_params["n"]
    prompts = [prompt_fn(problem["question"]) for problem in problems]

    outputs = llm.generate_completions(prompts, eval_sampling_params)
    if len(outputs) != len(problems) * n:
        raise ValueError(f"Expected {len(problems) * n} outputs, got {len(outputs)}.")

    for i, problem in enumerate(problems):
        problem_outputs = outputs[i * n : (i + 1) * n]
        generated_texts = [output.text for output in problem_outputs]
        last_answer_line = problem["answer"].rstrip("\n").split("\n")[-1]
        answer = last_answer_line.removeprefix("####").strip()

        problem["generated_response"] = generated_texts

        tmp = [reward_fn(text, answer) for text in generated_texts]
        problem["format_reward"] = [x["format_reward"] for x in tmp]
        problem["reward"] = [x["reward"] for x in tmp]


## 0-shot prompting

In [ ]:
sp = dict(
    temperature=0.05,
    n=1,
    max_tokens=512,
    seed=0,
    return_token_ids=True,
    stop=["</answer>"],
    include_stop_str_in_output=True
)
problems = test_data
# evaluate_llm(llm, r1_zero_reward_fn, r1zero_prompt, problems, sp)
# evaluate_llm(llm, r1_zero_reward_fn, question_only_prompt, problems, sp)
evaluate_llm(llm, r1_zero_reward_fn, r1zero_3shot_prompt, problems, sp)


(APIServer pid=422663) INFO:     127.0.0.1:60086 - "POST /v1/completions HTTP/1.1" 200 OK


In [8]:
rewards = [p["reward"][0] for p in problems]

print(f"{int(sum(rewards))}/{len(rewards)} answers are correct.")

530/1319 answers are correct.


## 3-shot prompting

In [ ]:
sp = dict(
    temperature=1.0,
    n=1,
    max_tokens=512,
    seed=0,
    return_token_ids=True,
    stop=["</answer>"],
    include_stop_str_in_output=True
)
problems = test_data
evaluate_llm(llm, r1_zero_reward_fn, r1zero_3shot_prompt, problems, sp)

(APIServer pid=69587) INFO:     127.0.0.1:60282 - "POST /v1/completions HTTP/1.1" 200 OK


In [ ]:
rewards = [p["reward"][0] for p in problems]

print(f"{int(sum(rewards))}/{len(rewards)} answers are correct.")

198/1319 answers are correct.


In [15]:
llm.stop()